In [7]:
# ================================================
# Marketplace Product Analytics & LTV Modeling
# Author: Mohammed Moniruzzaman Khan
# PhD Candidate in Mathematics
# ================================================

# -------------------------------
# 0️⃣ Imports
# -------------------------------
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# -------------------------------
# 1️⃣ Reproducibility & Paths
# -------------------------------
np.random.seed(42)

DATA_PATH = "data"
SQL_PATH = "sql/warehouse.db"
OUTPUT_PLOTS = "outputs/plots"
OUTPUT_DASHBOARDS = "outputs/dashboards"
OUTPUT_TABLES = "outputs/tables"
OUTPUT_REPORTS = "outputs/reports"

# Create folders if they do not exist
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs("sql", exist_ok=True)
os.makedirs(OUTPUT_PLOTS, exist_ok=True)
os.makedirs(OUTPUT_DASHBOARDS, exist_ok=True)
os.makedirs(OUTPUT_TABLES, exist_ok=True)
os.makedirs(OUTPUT_REPORTS, exist_ok=True)

# -------------------------------
# 2️⃣ Simulate Users & Transactions
# -------------------------------
n_users = 1000
users = pd.DataFrame({
    "user_id": np.arange(1, n_users+1),
    "cohort": np.random.choice(["Jan", "Feb", "Mar"], size=n_users)
})
users.to_csv(f"{DATA_PATH}/users.csv", index=False)

n_orders = 3000
transactions = pd.DataFrame({
    "transaction_id": np.arange(1, n_orders+1),
    "user_id": np.random.choice(users["user_id"], size=n_orders),
    "amount": np.round(np.random.exponential(50, n_orders), 2),
    "month": np.random.choice(["Jan", "Feb", "Mar"], size=n_orders),
    "treatment": np.random.choice([0,1], size=n_orders)
})
transactions.to_csv(f"{DATA_PATH}/transactions.csv", index=False)

# -------------------------------
# 3️⃣ SQL Warehouse Simulation
# -------------------------------
conn = sqlite3.connect(SQL_PATH)
users.to_sql("users", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)

# KPI summary per cohort using SQL JOIN to include cohort
kpi_query = """
SELECT u.cohort,
       COUNT(DISTINCT t.user_id) AS buyers,
       SUM(t.amount) AS revenue,
       AVG(t.amount) AS AOV
FROM transactions t
JOIN users u ON t.user_id = u.user_id
GROUP BY u.cohort
"""
kpi_summary = pd.read_sql(kpi_query, conn)
kpi_summary.to_csv(f"{OUTPUT_TABLES}/kpi_sql_summary.csv", index=False)

# -------------------------------
# 4️⃣ A/B Experimentation
# -------------------------------
treatment = transactions[transactions["treatment"]==1]
control = transactions[transactions["treatment"]==0]

# Conversion Rates
cr_treat = treatment["user_id"].nunique() / n_users
cr_control = control["user_id"].nunique() / n_users

# Two-sample t-test on total spend per user
t_stat, p_val = stats.ttest_ind(
    treatment.groupby("user_id")["amount"].sum(),
    control.groupby("user_id")["amount"].sum(),
    equal_var=False
)

# Conversion lift %
lift_percent = (cr_treat - cr_control) / cr_control * 100

# Save A/B results
with open(f"{OUTPUT_REPORTS}/ab_test_results.txt", "w") as f:
    f.write(f"CR Treatment: {cr_treat:.4f}, CR Control: {cr_control:.4f}\n")
    f.write(f"t-stat: {t_stat:.4f}, p-value: {p_val:.4f}\n")
    f.write(f"Lift (%): {lift_percent:.2f}\n")

# -------------------------------
# 5️⃣ Retention Cohort Analysis
# -------------------------------
cohort_table = pd.crosstab(
    users['cohort'],
    transactions['month'],
    values=transactions['transaction_id'],
    aggfunc='count',
    normalize='index'
).fillna(0)
cohort_table.to_csv(f"{OUTPUT_TABLES}/retention_cohort.csv")

# Plot cohort heatmap
plt.figure(figsize=(8,5))
sns.heatmap(cohort_table, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Retention Cohort Heatmap")
plt.savefig(f"{OUTPUT_PLOTS}/retention_cohort_heatmap.png")
plt.close()

# -------------------------------
# 6️⃣ Marketplace LTV Modeling (Stochastic)
# -------------------------------
H = 12  # months
purchase_freq = transactions.groupby("user_id")["transaction_id"].count()
aov_user = transactions.groupby("user_id")["amount"].mean()

# Monte Carlo simulations
simulations = 1000
ltv_samples_list = []

for i in range(simulations):
    freq_sim = np.random.poisson(purchase_freq)
    aov_sim = np.random.normal(aov_user, aov_user*0.1)
    # Convert to Series to avoid concat errors
    ltv_samples_list.append(pd.Series(freq_sim * aov_sim * H, index=purchase_freq.index))

ltv_samples = pd.concat(ltv_samples_list, axis=1)
ltv_samples.columns = [f"sim_{i+1}" for i in range(simulations)]

# Compute mean & 95% confidence interval
ltv_mean = ltv_samples.mean(axis=1)
ltv_lower = ltv_samples.quantile(0.025, axis=1)
ltv_upper = ltv_samples.quantile(0.975, axis=1)

# Treatment assignment example: Jan cohort = treatment
treatment_map = users.set_index("user_id")["cohort"].map(lambda x: 1 if x=="Jan" else 0)

# Combine results
ltv_estimates = pd.DataFrame({
    "user_id": purchase_freq.index,
    "LTV_mean": ltv_mean,
    "LTV_lower": ltv_lower,
    "LTV_upper": ltv_upper
})
ltv_estimates = ltv_estimates.join(treatment_map.rename("treatment"))
ltv_estimates.to_csv(f"{OUTPUT_TABLES}/ltv_estimates.csv", index=False)

# -------------------------------
# 7️⃣ Executive Dashboard
# -------------------------------
fig, axs = plt.subplots(2,2, figsize=(12,10))

# Conversion Rate
axs[0,0].bar(["Control","Treatment"], [cr_control, cr_treat], color=['skyblue','orange'])
axs[0,0].set_title("Conversion Rate")

# Average Order Value by cohort
axs[0,1].bar(kpi_summary['cohort'], kpi_summary['AOV'], color='skyblue')
axs[0,1].set_title("Average Order Value (AOV)")

# Revenue by cohort
axs[1,0].bar(kpi_summary['cohort'], kpi_summary['revenue'], color='green')
axs[1,0].set_title("Total Revenue")

# LTV by treatment/control
ltv_treat = ltv_estimates[ltv_estimates["treatment"]==1]["LTV_mean"].mean()
ltv_control = ltv_estimates[ltv_estimates["treatment"]==0]["LTV_mean"].mean()
axs[1,1].bar(["Control","Treatment"], [ltv_control, ltv_treat], color=['purple','red'])
axs[1,1].set_title("Mean LTV")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DASHBOARDS}/full_product_dashboard.png")
plt.close()

print("✅ Full Marketplace Analytics pipeline executed successfully. All outputs saved in /outputs/")

✅ Full Marketplace Analytics pipeline executed successfully. All outputs saved in /outputs/
